# Phase 1 - RAG Tool Explore

강의 PDF를 로드하고 Chroma 벡터스토어에 저장한 뒤, `search_lecture_materials` 도구로 검색되는지 확인합니다.

완료 기준:
- `PDFs/*.pdf` 전체 로드
- 검색어에 관련 강의 자료 텍스트 반환
- `search_lecture_materials.invoke(...)` 단독 호출 성공

## 1. 환경 로드

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "practice":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

load_dotenv(PROJECT_ROOT / ".env")

upstage_key = os.getenv("UPSTAGE_API_KEY")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("UPSTAGE_API_KEY loaded:", bool(upstage_key))
if upstage_key:
    print("UPSTAGE_API_KEY preview:", upstage_key[:8] + "...")

## 2. PDF 로드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_DIR = PROJECT_ROOT.parent / "PDFs"
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))

docs = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

print("PDF_DIR:", PDF_DIR)
print(f"총 {len(pdf_paths)}개 PDF, {len(docs)}개 페이지 로드")
print(docs[0].metadata if docs else "로드된 문서 없음")

## 3. 청크 분할

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"총 {len(chunks)}개 청크")
print(chunks[0].page_content[:300] if chunks else "생성된 청크 없음")

## 4. Chroma 저장

In [ ]:
from langchain_chroma import Chroma
from langchain_upstage import UpstageEmbeddings

CHROMA_DIR = str((PROJECT_ROOT / "data" / "chroma_db").resolve())
COLLECTION_NAME = "lecture_materials"

passage_embeddings = UpstageEmbeddings(model="solar-embedding-1-passage")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=passage_embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print("Chroma 저장 완료:", CHROMA_DIR)

## 5. 검색 테스트

In [ ]:
query_embeddings = UpstageEmbeddings(model="solar-embedding-1-query")
query_vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=query_embeddings,
)
retriever = query_vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("RAG 파이프라인 구성 방법")
for result in results:
    source = Path(result.metadata.get("source", "unknown")).name
    page = result.metadata.get("page", "?")
    print(f"[{source} p.{page}]")
    print(result.page_content[:300])
    print("---")

## 6. Tool 래핑

In [ ]:
from langchain_core.tools import tool
from my_project.api_limits import create_query_limiter

query_limiter = create_query_limiter()


@tool
def search_lecture_materials(query: str) -> str:
    """강의 자료에서 LangChain 파이프라인 관련 내용을 검색합니다.
    RAG, Agent, 멀티에이전트, 메모리, LangGraph 등의 패턴을 찾을 때 사용하세요.
    """
    try:
        docs = query_limiter.run("upstage_embedding", lambda: retriever.invoke(query))
        if not docs:
            return "강의 자료에서 관련 내용을 찾을 수 없습니다."

        formatted = []
        for doc in docs:
            source = Path(doc.metadata.get("source", "unknown")).name
            page = doc.metadata.get("page", "?")
            formatted.append(f"[{source} p.{page}]\n{doc.page_content[:500]}")
        return "\n\n".join(formatted)
    except Exception as exc:
        return f"강의 자료 검색 실패: {exc}"


print(search_lecture_materials.invoke("에이전트와 도구 사용법"))